In [3]:
from pyspark.sql.functions import when, col, udf
from pyspark.sql.types import StringType
# ensure the below library is installed on your fabric environment
import reverse_geocoder as rg


StatementMeta(, bcf3eb10-f03a-4af8-a056-0100cbcb5c12, 10, Finished, Available, Finished, False)

In [ ]:
# install reverse_geocoder -> %pip install reverse_geocoder 

StatementMeta(, bcf3eb10-f03a-4af8-a056-0100cbcb5c12, 8, Finished, Available, Finished, True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 11.9 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... - \ done
  Created wheel for reverse_geocoder: filename=reverse_geocoder-1.5.1-py3-none-any.whl size=2268064 sha256=57310454849c7e4c6d1ad0b9b77fb3071caa22cbfcff51fc3d44b54fe025ce13
  Stored in directory: /home/trusted-service-user/.cache/pip/wheels/17/3c/41/2bc89719586c2a5c53e9a527daa76a968a1288315c1ae2d904
Successfully built reverse_geocoder

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [4]:
def get_country_code(lat, lon):
    """
    Retrieve the country code for a given latitude and longitude.

    Parameters:
    lat (float or str): Latitude of the location.
    lon (float or str): Longitude of the location.

    Returns:
    str: Country code of the location, retrieved using the reverse geocoding API.

    Example:
    >>> get_country_details(48.8588443, 2.2943506)
    'FR'
    """
    coordinates = (float(lat), float(lon))
    return rg.search(coordinates)[0].get('cc')

StatementMeta(, bcf3eb10-f03a-4af8-a056-0100cbcb5c12, 11, Finished, Available, Finished, False)

In [5]:

# registering the udfs so they can be used on spark dataframes
get_country_code_udf = udf(get_country_code, StringType())

StatementMeta(, bcf3eb10-f03a-4af8-a056-0100cbcb5c12, 12, Finished, Available, Finished, False)

In [7]:
# adding country_code and city attributes
df = spark.read.table("DE_Learning_LH.dbo.earth_quake_silver")
df_with_location = df.withColumn(
    "country_code", 
    get_country_code_udf(col("latitude"), col("longtiude"))
)
     

StatementMeta(, bcf3eb10-f03a-4af8-a056-0100cbcb5c12, 14, Finished, Available, Finished, False)

In [8]:
# adding significance classification
df_with_location_sig_class = df_with_location.\
                                withColumn('sig_class', 
                                            when(col("sig") < 100, "Low").\
                                            when((col("sig") >= 100) & (col("sig") < 500), "Moderate").\
                                            otherwise("High")
                                            )

StatementMeta(, bcf3eb10-f03a-4af8-a056-0100cbcb5c12, 15, Finished, Available, Finished, False)

In [9]:
# appending the data to the gold table
df_with_location_sig_class.write.mode('append').saveAsTable('earth_quake_gold')

StatementMeta(, bcf3eb10-f03a-4af8-a056-0100cbcb5c12, 16, Finished, Available, Finished, False)

In [10]:
df = spark.sql("SELECT * FROM DE_Learning_LH.dbo.earth_quake_gold LIMIT 5")
display(df)

StatementMeta(, bcf3eb10-f03a-4af8-a056-0100cbcb5c12, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c135404d-9f4f-4237-a245-0c39a41609d0)